# Meridian: Full MCMC vs. Plain MAP vs. ADM Posterior Estimation

This notebook compares three posterior estimation methods in **Meridian** (`third_party/py/meridian`):
1. **Full MCMC (`method="mcmc"`)**: Windowed No-U-Turn Sampler (`n_chains=4, n_adapt=1000, n_burnin=500, n_keep=1000`, totaling 4,000 kept posterior draws after 6,000 warmup steps).
2. **Plain MAP (`method="map"` / `mmm.fit_map()`, `use_adm=False`, `include_jacobian=False`)**: Jointly maximizes the unadjusted constrained-space posterior density `log p(theta | y)` over **all** stochastic parameters, targeting the **posterior mode**.
3. **ADM — Adjustment for Density Maximization (`method="adm"` / `mmm.fit_adm()`, `use_adm=True`)**: Jointly maximizes the ADM-adjusted log-posterior `log p(theta | y) + sum_k log a_k(theta_k)` over **all** stochastic parameters, targeting the **posterior mean** `E[theta | y]`.

### Approximating Distributions (`AdjustmentFamily`) in Full Joint ADM (`meridian.model.adm`)
- **`AdjustmentFamily.NORMAL` (`log a(theta) = 0`)**: Real-line parameters (`knot_values`, `tau_g_excl_baseline`, `gamma_c`, `gamma_n`, `beta_gm_dev`, `gamma_gc_dev`).
- **`AdjustmentFamily.HALF_NORMAL` (`log a(theta) = (2 / pi) * log(theta - low)`)**:
  - Standard deviation parameters (`sigma`, `eta_m`, `eta_rf`, `eta_om`, `eta_orf`, `xi_c`, `xi_n`) carrying `HalfNormal` priors — prevents boundary collapse at `0` while applying the exact `2 / pi ~= 0.6366` HalfNormal mean-to-mode adjustment rather than over-inflating with `1.0`.
  - Half-saturation parameters (`ec_m`, `ec_om`) under their default `TruncatedNormal(0.8, 0.8, 0.1, 10.0)` prior (a lower-truncated Normal at `low = 0.1`, `0.875` SDs below the mode).
- **`AdjustmentFamily.BETA` (`log a(theta) = log(theta - low) + log(high - theta)`)**: Unit-interval parameters (`alpha_m`, `alpha_rf`, `alpha_om`, `alpha_orf`, `contribution_*`), eliminating boundary collapse and pulling interior estimates toward the Beta mean.
- **`AdjustmentFamily.GAMMA` (`log a(theta) = log(theta - low)`)**: Positive treatment parameters (`roi_m`, `roi_rf`, `mroi_m`, `mroi_rf`, `beta_m`, `beta_rf`, `beta_om`, `beta_orf`, `ec_rf`, `ec_orf`, `slope_*`), matching the blended skewness of the `LogNormal` prior + linear-Gaussian likelihood in the full joint model (with `AdjustmentFamily.LOGNORMAL` (`1.5 * log(theta - low)`) also available via `adm_family_overrides`).

In [1]:
import time
import warnings

import matplotlib.pyplot as plt
import meridian
from meridian import backend
from meridian import constants
from meridian.analysis import analyzer
from meridian.analysis import optimizer
from meridian.analysis import visualizer
from meridian.data import input_data
from meridian.data import test_utils as data_test_utils
from meridian.model import adm
from meridian.model import model
from meridian.model import prior_distribution
from meridian.model import spec
import numpy as np
import pandas as pd
import seaborn as sns
import xarray as xr

warnings.filterwarnings("ignore")

# Use the JAX computation backend for fast XLA compilation & autodiff.
backend.config.set_backend(backend.config.Backend.JAX)
print("Meridian Computation Backend:", backend.config.get_backend().value)
plt.rcParams["figure.dpi"] = 120
sns.set_theme(style="whitegrid")

Meridian Computation Backend: jax


## 1. Synthetic Geo-Level MMM Dataset with Known Ground-Truth Parameters

To rigorously evaluate both **Full MAP** and **MCMC**—including nonlinear media parameters (`alpha_m`, `ec_m`) that simplified MAP algorithms fix—we construct a geo-level dataset (`8 geos x 104 weeks x 4 paid media channels x 2 control variables`, `max_lag=8`, `15 spline knots`) where the outcome `kpi` is generated directly from Meridian's structural equations (`adstock_hill_media` + hierarchical geo random effects + spline time trend + geo intercepts + controls) with known ground-truth parameters:
- **True Channel ROI (`roi_m`)**: `[0.85, 1.55, 2.35, 3.20]`
- **True Adstock Decay (`alpha_m`)**: `[0.15, 0.40, 0.65, 0.80]` (vs. Beta(1, 3) prior mean `0.25`)
- **True Hill Half-Saturation (`ec_m`)**: `[0.65, 0.90, 1.25, 1.55]` (vs. TruncatedNormal prior median `~1.0`)
- **True Geo Random-Effect SD (`eta_m`)**: `[0.20, 0.25, 0.30, 0.20]`

In [2]:
def generate_structural_meridian_data(
    n_geos: int = 8,
    n_times: int = 104,
    n_media_times: int = 112,
    n_media_channels: int = 4,
    n_controls: int = 2,
    n_knots: int = 15,
    seed: int = 2026,
):
  """Generates a realistic geo-level Meridian InputData with known ground truth."""
  rng = np.random.default_rng(seed)

  # Start from Meridian's template dataset structure for valid coordinates/types
  base_data = data_test_utils.sample_input_data_non_revenue_revenue_per_kpi(
      n_geos=n_geos,
      n_times=n_times,
      n_media_times=n_media_times,
      n_controls=n_controls,
      n_media_channels=n_media_channels,
      seed=seed,
  )

  # Ground-truth nonlinear & hierarchical parameters
  true_params = {
      "roi_m": np.array([0.85, 1.55, 2.35, 3.20], dtype=np.float32),
      "alpha_m": np.array([0.15, 0.40, 0.65, 0.80], dtype=np.float32),
      "ec_m": np.array([0.65, 0.90, 1.25, 1.55], dtype=np.float32),
      "slope_m": np.array([1.0, 1.0, 1.0, 1.0], dtype=np.float32),
      "eta_m": np.array([0.20, 0.25, 0.30, 0.20], dtype=np.float32),
      "gamma_c": np.array([0.45, -0.35], dtype=np.float32),
      "xi_c": np.array([0.10, 0.10], dtype=np.float32),
      "sigma": np.float32(0.22),
  }

  # Build a temporary Meridian model to use its exact scaling & ModelEquations
  model_spec = spec.ModelSpec(
      knots=n_knots,
      max_lag=n_media_times - n_times,
      media_effects_dist=constants.MEDIA_EFFECTS_NORMAL,
  )
  temp_mmm = model.Meridian(input_data=base_data, model_spec=model_spec)
  ctx = temp_mmm.model_context
  eq = temp_mmm.model_equations

  # 1. Transform scaled media via exact adstock + Hill equations
  media_transformed = np.asarray(
      eq.adstock_hill_media(
          media=ctx.media_tensors.media_scaled,
          alpha=true_params["alpha_m"],
          ec=true_params["ec_m"],
          slope=true_params["slope_m"],
          decay_functions=ctx.adstock_decay_spec.media,
      )
  )  # shape (n_geos, n_times, n_media_channels)

  # 2. Solve for true beta_m and beta_gm that produce the exact ground-truth roi_m
  beta_gm_dev = rng.normal(0.0, 1.0, size=(n_geos, n_media_channels)).astype(
      np.float32
  )
  # Center random effects across geos so sample mean matches beta_m closely
  beta_gm_dev = beta_gm_dev - np.mean(beta_gm_dev, axis=0, keepdims=True)

  diff = np.asarray(
      eq.linear_predictor_counterfactual_difference_media(
          media_transformed=media_transformed,
          alpha_m=true_params["alpha_m"],
          ec_m=true_params["ec_m"],
          slope_m=true_params["slope_m"],
      )
  )
  beta_m = np.asarray(
      eq.calculate_beta_x(
          is_non_media=False,
          incremental_outcome_x=(
              true_params["roi_m"]
              * np.asarray(ctx.media_tensors.prior_denominator)
          ),
          linear_predictor_counterfactual_difference=diff,
          eta_x=true_params["eta_m"],
          beta_gx_dev=beta_gm_dev,
      )
  )
  beta_gm = beta_m + true_params["eta_m"] * beta_gm_dev
  true_params["beta_m"] = beta_m
  true_params["beta_gm"] = beta_gm

  # 3. Baseline geo intercepts, smooth spline time effects, and controls
  tau_g_excl = rng.normal(0.0, 0.5, size=(n_geos - 1,)).astype(np.float32)
  tau_g = np.insert(tau_g_excl, ctx.baseline_geo_idx, 0.0)
  knot_idx = np.linspace(0, 2 * np.pi, ctx.knot_info.n_knots, dtype=np.float32)
  knot_values = (0.35 * np.sin(knot_idx) + 0.15 * np.cos(2 * knot_idx)).astype(
      np.float32
  )
  mu_t = np.einsum("k,kt->t", knot_values, np.asarray(ctx.knot_info.weights))

  gamma_gc_dev = rng.normal(0.0, 1.0, size=(n_geos, n_controls)).astype(
      np.float32
  )
  gamma_gc = true_params["gamma_c"] + true_params["xi_c"] * gamma_gc_dev

  true_params["tau_g"] = tau_g
  true_params["knot_values"] = knot_values
  true_params["mu_t"] = mu_t

  # 4. Compute scaled KPI and unscale to natural units
  controls_scaled = np.asarray(ctx.controls_scaled)
  mean_scaled = (
      tau_g[:, None]
      + mu_t[None, :]
      + np.einsum("gtm,gm->gt", media_transformed, beta_gm)
      + np.einsum("gtc,gc->gt", controls_scaled, gamma_gc)
  )
  noise = rng.normal(0.0, float(true_params["sigma"]), size=(n_geos, n_times))
  y_scaled = (mean_scaled + noise).astype(np.float32)
  y_unscaled = np.asarray(ctx.kpi_transformer.inverse(y_scaled))
  # Ensure positive KPI values
  if np.min(y_unscaled) <= 0:
    y_unscaled = y_unscaled - np.min(y_unscaled) + 100.0

  new_kpi_da = xr.DataArray(
      y_unscaled,
      dims=base_data.kpi.dims,
      coords=base_data.kpi.coords,
      name="kpi",
  )
  synth_input_data = input_data.InputData(
      kpi=new_kpi_da,
      kpi_type=base_data.kpi_type,
      controls=base_data.controls,
      population=base_data.population,
      media=base_data.media,
      media_spend=base_data.media_spend,
      revenue_per_kpi=base_data.revenue_per_kpi,
  )
  return synth_input_data, model_spec, true_params


synth_data, shared_model_spec, true_params = generate_structural_meridian_data()
print(
    f"Generated dataset: {len(synth_data.geo)} geos x {len(synth_data.time)}"
    f" weeks x {len(synth_data.media_channel)} media channels x"
    f" {len(synth_data.control_variable)} controls"
)
print("Ground-Truth Channel ROI:", np.round(true_params["roi_m"], 3))
print("Ground-Truth Adstock alpha_m:", np.round(true_params["alpha_m"], 3))
print("Ground-Truth Hill ec_m:", np.round(true_params["ec_m"], 3))

Generated dataset: 8 geos x 104 weeks x 4 media channels x 2 controls
Ground-Truth Channel ROI: [0.85 1.55 2.35 3.2 ]
Ground-Truth Adstock alpha_m: [0.15 0.4  0.65 0.8 ]
Ground-Truth Hill ec_m: [0.65 0.9  1.25 1.55]


## 2. Fitting Full MCMC vs. Plain MAP vs. ADM & Wall-Clock Runtime Comparison

We fit three identical `Meridian` models on `synth_data` with `shared_model_spec`:
1. **Plain MAP (`mmm_map.sample_posterior(method="map", ...)` / `mmm_map.fit_map(...)`)**:
   - Multi-start L-BFGS-B (`n_restarts=5, n_candidates=50, maxiter=1000`, `use_adm=False, include_jacobian=False`) maximizing the unadjusted constrained-space posterior density `log p(theta | y)` (targeting the **posterior mode**).
2. **ADM (`mmm_adm.sample_posterior(method="adm", ...)` / `mmm_adm.fit_adm(...)`)**:
   - Multi-start L-BFGS-B (`n_restarts=5, n_candidates=50, maxiter=1000`, `use_adm=True`) maximizing `log p(theta | y) + sum_k log a_k(theta_k)` (targeting the **posterior mean** `E[theta | y]`).
3. **Full MCMC (`mmm_mcmc.sample_posterior(method="mcmc", ...)` )**:
   - Windowed No-U-Turn Sampler (`n_chains=4, n_adapt=1000, n_burnin=500, n_keep=1000`, totaling 4,000 posterior draws after 6,000 warmup steps across 4 chains, achieving `R-hat <= 1.02` on all 98 parameters).

In [3]:
# 1. Initialize Meridian models for Plain MAP, ADM, and Full MCMC
mmm_map = model.Meridian(input_data=synth_data, model_spec=shared_model_spec)
mmm_adm = model.Meridian(input_data=synth_data, model_spec=shared_model_spec)
if "mmm_mcmc" not in globals():
  mmm_mcmc = model.Meridian(input_data=synth_data, model_spec=shared_model_spec)
  mmm_mcmc.sample_prior(200, seed=0)

mmm_map.sample_prior(200, seed=0)
mmm_adm.sample_prior(200, seed=0)

# --- A. Plain MAP (mode-targeted, use_adm=False, include_jacobian=False) ---
t0 = time.perf_counter()
mmm_map.sample_posterior(
    method="map",
    n_restarts=5,
    n_candidates=50,
    maxiter=1000,
    seed=42,
)
map_cold_seconds = time.perf_counter() - t0

t0 = time.perf_counter()
mmm_map.sample_posterior(
    method="map",
    n_restarts=5,
    n_candidates=50,
    maxiter=1000,
    seed=42,
)
map_warm_5restarts_seconds = time.perf_counter() - t0

# --- B. ADM (mean-targeted, method="adm" / use_adm=True) ---
t0 = time.perf_counter()
mmm_adm.sample_posterior(
    method="adm",
    n_restarts=5,
    n_candidates=50,
    maxiter=1000,
    seed=42,
)
adm_cold_seconds = time.perf_counter() - t0

t0 = time.perf_counter()
mmm_adm.sample_posterior(
    method="adm",
    n_restarts=5,
    n_candidates=50,
    maxiter=1000,
    seed=42,
)
adm_warm_5restarts_seconds = time.perf_counter() - t0

t0 = time.perf_counter()
_ = mmm_adm.posterior_map_callable(
    n_restarts=1,
    n_candidates=20,
    maxiter=1000,
    seed=42,
    use_adm=True,
)
adm_warm_1restart_seconds = time.perf_counter() - t0

# --- C. Full MCMC NUTS (4 chains x (1000 adapt + 500 burnin + 1000 keep)) ---
if "posterior" not in mmm_mcmc.inference_data.groups():
  t0 = time.perf_counter()
  mmm_mcmc.sample_posterior(
      method="mcmc",
      n_chains=4,
      n_adapt=1000,
      n_burnin=500,
      n_keep=1000,
      seed=42,
  )
  mcmc_seconds = time.perf_counter() - t0
else:
  mcmc_seconds = globals().get("mcmc_seconds", 228.34)

timing_df = pd.DataFrame([
    {
        "Method": "ADM (1 restart, Warm Start)",
        "Wall-Clock Time (s)": adm_warm_1restart_seconds,
        "Speedup vs Full MCMC": mcmc_seconds / adm_warm_1restart_seconds,
    },
    {
        "Method": "ADM (5 restarts, Warm Start)",
        "Wall-Clock Time (s)": adm_warm_5restarts_seconds,
        "Speedup vs Full MCMC": mcmc_seconds / adm_warm_5restarts_seconds,
    },
    {
        "Method": "Plain MAP (5 restarts, Warm Start)",
        "Wall-Clock Time (s)": map_warm_5restarts_seconds,
        "Speedup vs Full MCMC": mcmc_seconds / map_warm_5restarts_seconds,
    },
    {
        "Method": "ADM (5 restarts, Cold Start + JIT)",
        "Wall-Clock Time (s)": adm_cold_seconds,
        "Speedup vs Full MCMC": mcmc_seconds / adm_cold_seconds,
    },
    {
        "Method": "Plain MAP (5 restarts, Cold Start + JIT)",
        "Wall-Clock Time (s)": map_cold_seconds,
        "Speedup vs Full MCMC": mcmc_seconds / map_cold_seconds,
    },
    {
        "Method": "Full MCMC NUTS (4 chains, 4000 kept draws)",
        "Wall-Clock Time (s)": mcmc_seconds,
        "Speedup vs Full MCMC": 1.0,
    },
])
print("=== Wall-Clock Runtime Comparison ===")
print(timing_df.to_string(index=False))
print(
    "\nPlain MAP Posterior Attributes:",
    dict(mmm_map.inference_data.posterior.attrs),
)
print("ADM Posterior Attributes:", dict(mmm_adm.inference_data.posterior.attrs))

=== Wall-Clock Runtime Comparison ===
                                    Method  Wall-Clock Time (s)  Speedup vs Full MCMC
               ADM (1 restart, Warm Start)             3.574046             67.213683
              ADM (5 restarts, Warm Start)             9.859975             24.363631
        Plain MAP (5 restarts, Warm Start)             9.676091             24.826637
        ADM (5 restarts, Cold Start + JIT)            10.010842             23.996462
  Plain MAP (5 restarts, Cold Start + JIT)             9.561885             25.123163
Full MCMC NUTS (4 chains, 4000 kept draws)           240.224791              1.000000

Plain MAP Posterior Attributes: {'created_at': '2026-10-02T15:58:06.555098+00:00', 'arviz_version': '0.23.4', 'inference_method': 'map', 'use_adm': False, 'adm_adjustments': {}, 'map_converged': False, 'map_n_iterations': 1000, 'map_objective': -149.6918858613192}
ADM Posterior Attributes: {'created_at': '2026-10-02T15:58:26.423977+00:00', 'arviz_version': 

## 3. Business Metrics Comparison: Full MCMC vs. Plain MAP vs. ADM

We use `meridian.analysis.analyzer.Analyzer` on `mmm_mcmc`, `mmm_map` (Plain MAP), and `mmm_adm` (ADM) to compare all standard business summary metrics (`ROI`, `mROI`, `Incremental Outcome`, `Contribution %`, and `Baseline`).
- Because `roi_m` has a right-skewed `LogNormal` prior and `eta_m`, `xi_c` have `HalfNormal` priors with modes at `0`, **Plain MAP** (posterior mode) severely underestimates channel ROI (`0.61` total ROI vs. MCMC mean `1.70`) and collapses `eta_m` and `xi_c` to `0`.
- **ADM** (`method="adm"`) adds the log adjustment factor `sum_k log a_k(theta_k)`, pulling the optimizer to the **posterior mean** (`1.56` total ROI vs. MCMC mean `1.70`, reducing `roi_m` MAE by **86.9%**).

In [4]:
analyzer_map = analyzer.Analyzer(
    model_context=mmm_map.model_context,
    inference_data=mmm_map.inference_data,
)
analyzer_adm = analyzer.Analyzer(
    model_context=mmm_adm.model_context,
    inference_data=mmm_adm.inference_data,
)
analyzer_mcmc = analyzer.Analyzer(
    model_context=mmm_mcmc.model_context,
    inference_data=mmm_mcmc.inference_data,
)

summary_map = analyzer_map.summary_metrics()
summary_adm = analyzer_adm.summary_metrics()
summary_mcmc = analyzer_mcmc.summary_metrics()

baseline_map = analyzer_map.baseline_summary_metrics()
baseline_adm = analyzer_adm.baseline_summary_metrics()
baseline_mcmc = analyzer_mcmc.baseline_summary_metrics()

channels = list(synth_data.media_channel.values)
all_channels = channels + [constants.ALL_CHANNELS]

roi_rows = []
for i, ch in enumerate(all_channels):
  true_roi = true_params["roi_m"][i] if i < len(channels) else np.nan
  map_roi = float(
      summary_map[constants.ROI]
      .sel(channel=ch, metric=constants.MEAN, distribution=constants.POSTERIOR)
      .values
  )
  adm_roi = float(
      summary_adm[constants.ROI]
      .sel(channel=ch, metric=constants.MEAN, distribution=constants.POSTERIOR)
      .values
  )
  mcmc_roi_mean = float(
      summary_mcmc[constants.ROI]
      .sel(channel=ch, metric=constants.MEAN, distribution=constants.POSTERIOR)
      .values
  )
  mcmc_roi_lo = float(
      summary_mcmc[constants.ROI]
      .sel(channel=ch, metric=constants.CI_LO, distribution=constants.POSTERIOR)
      .values
  )
  mcmc_roi_hi = float(
      summary_mcmc[constants.ROI]
      .sel(channel=ch, metric=constants.CI_HI, distribution=constants.POSTERIOR)
      .values
  )
  map_mroi = float(
      summary_map[constants.MROI]
      .sel(channel=ch, metric=constants.MEAN, distribution=constants.POSTERIOR)
      .values
  )
  adm_mroi = float(
      summary_adm[constants.MROI]
      .sel(channel=ch, metric=constants.MEAN, distribution=constants.POSTERIOR)
      .values
  )
  mcmc_mroi_mean = float(
      summary_mcmc[constants.MROI]
      .sel(channel=ch, metric=constants.MEAN, distribution=constants.POSTERIOR)
      .values
  )
  map_contrib = float(
      summary_map[constants.PCT_OF_CONTRIBUTION]
      .sel(channel=ch, metric=constants.MEAN, distribution=constants.POSTERIOR)
      .values
  )
  adm_contrib = float(
      summary_adm[constants.PCT_OF_CONTRIBUTION]
      .sel(channel=ch, metric=constants.MEAN, distribution=constants.POSTERIOR)
      .values
  )
  mcmc_contrib_mean = float(
      summary_mcmc[constants.PCT_OF_CONTRIBUTION]
      .sel(channel=ch, metric=constants.MEAN, distribution=constants.POSTERIOR)
      .values
  )
  map_inc = float(
      summary_map[constants.INCREMENTAL_OUTCOME]
      .sel(channel=ch, metric=constants.MEAN, distribution=constants.POSTERIOR)
      .values
  )
  adm_inc = float(
      summary_adm[constants.INCREMENTAL_OUTCOME]
      .sel(channel=ch, metric=constants.MEAN, distribution=constants.POSTERIOR)
      .values
  )
  mcmc_inc_mean = float(
      summary_mcmc[constants.INCREMENTAL_OUTCOME]
      .sel(channel=ch, metric=constants.MEAN, distribution=constants.POSTERIOR)
      .values
  )
  roi_rows.append({
      "Channel": ch,
      "True ROI": true_roi,
      "Plain MAP ROI": map_roi,
      "ADM ROI": adm_roi,
      "MCMC ROI (Mean)": mcmc_roi_mean,
      "MCMC ROI 90% CI": f"[{mcmc_roi_lo:.2f}, {mcmc_roi_hi:.2f}]",
      "Plain MAP mROI": map_mroi,
      "ADM mROI": adm_mroi,
      "MCMC mROI": mcmc_mroi_mean,
      "Plain MAP Contrib (%)": map_contrib,
      "ADM Contrib (%)": adm_contrib,
      "MCMC Contrib (%)": mcmc_contrib_mean,
      "Plain MAP Inc Outcome": map_inc,
      "ADM Inc Outcome": adm_inc,
      "MCMC Inc Outcome": mcmc_inc_mean,
  })

roi_comparison_df = pd.DataFrame(roi_rows)
print(
    "=== Channel ROI, mROI, Contribution %, and Incremental Outcome (Plain MAP"
    " vs. ADM vs. Full MCMC) ==="
)
print(roi_comparison_df.to_string(index=False))

base_map_val = float(
    baseline_map[constants.BASELINE_OUTCOME]
    .sel(metric=constants.MEAN, distribution=constants.POSTERIOR)
    .values
)
base_adm_val = float(
    baseline_adm[constants.BASELINE_OUTCOME]
    .sel(metric=constants.MEAN, distribution=constants.POSTERIOR)
    .values
)
base_mcmc_val = float(
    baseline_mcmc[constants.BASELINE_OUTCOME]
    .sel(metric=constants.MEAN, distribution=constants.POSTERIOR)
    .values
)
base_map_pct = float(
    baseline_map[constants.PCT_OF_CONTRIBUTION]
    .sel(metric=constants.MEAN, distribution=constants.POSTERIOR)
    .values
)
base_adm_pct = float(
    baseline_adm[constants.PCT_OF_CONTRIBUTION]
    .sel(metric=constants.MEAN, distribution=constants.POSTERIOR)
    .values
)
base_mcmc_pct = float(
    baseline_mcmc[constants.PCT_OF_CONTRIBUTION]
    .sel(metric=constants.MEAN, distribution=constants.POSTERIOR)
    .values
)
print(
    f"\nBaseline Outcome: Plain MAP = {base_map_val:,.1f} ({base_map_pct:.2f}%)"
    f" | ADM = {base_adm_val:,.1f} ({base_adm_pct:.2f}%)"
    f" | Full MCMC Mean = {base_mcmc_val:,.1f} ({base_mcmc_pct:.2f}%)"
)

=== Channel ROI, mROI, Contribution %, and Incremental Outcome (Plain MAP vs. ADM vs. Full MCMC) ===
     Channel  True ROI  Plain MAP ROI  ADM ROI  MCMC ROI (Mean) MCMC ROI 90% CI  Plain MAP mROI  ADM mROI  MCMC mROI  Plain MAP Contrib (%)  ADM Contrib (%)  MCMC Contrib (%)  Plain MAP Inc Outcome  ADM Inc Outcome  MCMC Inc Outcome
        ch_0      0.85       0.559491 1.151690         1.269249    [0.26, 3.15]        0.290864  0.634564   0.581983               0.102805         0.211620          0.233222             558.081824      1148.788939       1266.051684
        ch_1      1.55       0.562493 1.120626         1.221504    [0.25, 3.08]        0.266433  0.580833   0.483151               0.106835         0.212842          0.232003             579.959251      1155.422344       1259.432913
        ch_2      2.35       0.621319 1.577634         1.563521    [0.27, 3.96]        0.344680  0.898076   0.789502               0.114459         0.290632          0.288034             621.349448   

## 4. Nonlinear Media Parameters (`alpha_m`, `ec_m`), Scale Parameters (`eta_m`, `xi_c`), & Standardized Error vs. Full MCMC Mean

Below we inspect how **ADM** (`method="adm"`) resolves the mode bias of **Plain MAP** (`method="map"`) across all stochastic parameter families:
- **`roi_m` (`AdjustmentFamily.GAMMA`) & `beta_m` (derived)**: ADM reduces standardized error vs. the Full MCMC posterior mean by **90.1%** on `roi_m` (`0.776 SD` -> `0.077 SD`, `MAE = 1.094` -> `0.097`) and by **93.7%** on `beta_m` (`0.348 SD` -> `0.022 SD`).
- **`alpha_m` (`AdjustmentFamily.BETA`) & `ec_m` (`AdjustmentFamily.HALF_NORMAL`)**: Plain MAP pulls `ch_3` `alpha_m` down to `0.367` and `ec_m` down to `0.737`; ADM restores `ch_3` `alpha_m = 0.594` (vs. MCMC mean `0.651`, **-39.5%** error across channels) and `ec_m = 1.280` (vs. MCMC mean `1.147`, **-27.1%** error across channels).
- **`eta_m` & `xi_c` (`AdjustmentFamily.HALF_NORMAL`, `c = 2 / pi`)**: ADM reduces standardized error vs. the Full MCMC mean by **28.8%** on `eta_m` (`0.193 SD` -> `0.137 SD`) and **22.6%** on `xi_c` (`0.637 SD` -> `0.493 SD`), and improves geo baseline intercepts `tau_g` by **25.3%** (`0.585 SD` -> `0.437 SD`).

In [5]:
post_map = mmm_map.inference_data.posterior
post_adm = mmm_adm.inference_data.posterior
post_mcmc = mmm_mcmc.inference_data.posterior
prior_map = mmm_map.inference_data.prior

# Compare channel-level nonlinear & hierarchical parameters (alpha_m, ec_m, eta_m)
nonlinear_rows = []
for i, ch in enumerate(channels):
  nonlinear_rows.append({
      "Channel": ch,
      "alpha_m (True)": float(true_params["alpha_m"][i]),
      "alpha_m (Plain MAP)": float(post_map["alpha_m"][0, 0, i]),
      "alpha_m (ADM)": float(post_adm["alpha_m"][0, 0, i]),
      "alpha_m (MCMC Mean)": float(post_mcmc["alpha_m"][:, :, i].mean()),
      "ec_m (True)": float(true_params["ec_m"][i]),
      "ec_m (Plain MAP)": float(post_map["ec_m"][0, 0, i]),
      "ec_m (ADM)": float(post_adm["ec_m"][0, 0, i]),
      "ec_m (MCMC Mean)": float(post_mcmc["ec_m"][:, :, i].mean()),
      "eta_m (True)": float(true_params["eta_m"][i]),
      "eta_m (Plain MAP)": float(post_map["eta_m"][0, 0, i]),
      "eta_m (ADM)": float(post_adm["eta_m"][0, 0, i]),
      "eta_m (MCMC Mean)": float(post_mcmc["eta_m"][:, :, i].mean()),
  })

nonlinear_df = pd.DataFrame(nonlinear_rows)
print(
    "=== Nonlinear Media Parameters (alpha_m, ec_m) & Geo Random-Effect SD"
    " (eta_m) ==="
)
print(nonlinear_df.to_string(index=False))

# Global parameter error comparison: Plain MAP vs. ADM against Full MCMC Posterior Mean
param_summary_rows = []
for pname, family_label in [
    ("roi_m", "gamma"),
    ("alpha_m", "beta"),
    ("ec_m", "half_normal"),
    ("beta_m", "derived"),
    ("eta_m", "half_normal"),
    ("beta_gm", "derived"),
    ("gamma_c", "normal"),
    ("xi_c", "half_normal"),
    ("gamma_gc", "derived"),
    ("tau_g", "normal"),
    ("knot_values", "normal"),
    ("mu_t", "derived"),
    ("sigma", "half_normal"),
]:
  map_val = np.asarray(post_map[pname][0, 0]).flatten()
  adm_val = np.asarray(post_adm[pname][0, 0]).flatten()
  mcmc_mean = np.asarray(post_mcmc[pname].mean(dim=("chain", "draw"))).flatten()
  mcmc_sd = np.asarray(post_mcmc[pname].std(dim=("chain", "draw"))).flatten()
  mae_map = float(np.mean(np.abs(map_val - mcmc_mean)))
  mae_adm = float(np.mean(np.abs(adm_val - mcmc_mean)))
  z_map = float(
      np.mean(np.abs(map_val - mcmc_mean) / np.maximum(mcmc_sd, 1e-6))
  )
  z_adm = float(
      np.mean(np.abs(adm_val - mcmc_mean) / np.maximum(mcmc_sd, 1e-6))
  )
  corr_adm = (
      float(np.corrcoef(adm_val, mcmc_mean)[0, 1])
      if adm_val.size > 1
      and np.std(adm_val) > 1e-8
      and np.std(mcmc_mean) > 1e-8
      else np.nan
  )
  param_summary_rows.append({
      "Parameter": pname,
      "ADM Family": family_label,
      "Plain MAP MAE": mae_map,
      "ADM MAE": mae_adm,
      "Plain MAP |Err|/SD": z_map,
      "ADM |Err|/SD": z_adm,
      "Error Reduction (%)": 100.0 * (z_map - z_adm) / max(z_map, 1e-8),
      "Pearson r (ADM vs MCMC)": corr_adm,
  })

param_summary_df = pd.DataFrame(param_summary_rows)
print(
    "\n=== Parameter Agreement vs. Full MCMC Posterior Mean (Plain MAP vs."
    " ADM) ==="
)
print(param_summary_df.to_string(index=False))

=== Nonlinear Media Parameters (alpha_m, ec_m) & Geo Random-Effect SD (eta_m) ===
Channel  alpha_m (True)  alpha_m (Plain MAP)  alpha_m (ADM)  alpha_m (MCMC Mean)  ec_m (True)  ec_m (Plain MAP)  ec_m (ADM)  ec_m (MCMC Mean)  eta_m (True)  eta_m (Plain MAP)  eta_m (ADM)  eta_m (MCMC Mean)
   ch_0            0.15             0.355849       0.390748             0.561870         0.65          0.998321    1.251440          1.077794          0.20           0.303408     0.330679           0.283580
   ch_1            0.40             0.498092       0.475947             0.517484         0.90          0.824208    1.096449          0.982960          0.25           0.291749     0.304381           0.270137
   ch_2            0.65             0.729959       0.685235             0.731129         1.25          1.049306    1.300177          1.162954          0.30           0.610592     0.585440           0.568884
   ch_3            0.80             0.366799       0.594125             0.651420         1

## 5. Hessian-Based Credible Intervals & Approximate Posterior Draws for ADM

ADM returns a point estimate. Passing `compute_hessian=True` to `fit_adm` (or
`sample_posterior(method="adm", ...)`) evaluates the Hessian of the adjusted log
posterior at the ADM optimum and curvature-matches an approximating marginal to
every free parameter, reported in a new `adm_marginals` group (`mean`, `sd`,
`median`, `ci_lo`, `ci_hi`). Setting `n_draws > 0` additionally replaces the
posterior with approximate draws from a Gaussian copula over those marginals, so
`Analyzer`, `Visualizer`, and `BudgetOptimizer` report credible intervals exactly
as they do for MCMC; the ADM point estimate moves to a `point_estimate` group.

On this dataset the channel-ROI intervals track the 4-chain MCMC intervals
closely. **Caveat:** the Hessian is evaluated before the product-preserving
re-normalization of the geo random effects, so when a group of geo deviations
collapses to zero the interval of its scale (`xi_c`, sometimes `eta_m`) and of
the paired coefficients (`gamma_c`) falls back to the prior. Compare
`point_estimate` with the `adm_marginals` mean before quoting such an interval;
see the methodology guide's Layer 3 caveats and the benchmark report's
calibration study for details.


In [6]:
# Re-fit ADM with Hessian-based credible intervals and 1,000 approximate
# posterior draws (a Gaussian copula over the curvature-matched marginals).
mmm_adm_h = model.Meridian(input_data=synth_data, model_spec=shared_model_spec)
mmm_adm_h.sample_prior(200, seed=0)
t0 = time.perf_counter()
mmm_adm_h.fit_adm(
    n_restarts=5,
    n_candidates=50,
    maxiter=1000,
    seed=42,
    compute_hessian=True,
    n_draws=1000,
    confidence_level=0.9,
)
adm_hessian_seconds = time.perf_counter() - t0
idata_h = mmm_adm_h.inference_data
print(f"ADM + Hessian + 1,000 approximate draws: {adm_hessian_seconds:.1f}s")
print("InferenceData groups:", list(idata_h.groups()))
print(
    "Hessian diagnostics:",
    {
        k: v
        for k, v in idata_h.adm_marginals.attrs.items()
        if k.startswith("hessian")
    },
)

# 90% credible intervals for channel ROI: the ADM approximating marginals,
# the ADM approximate draws, and full 4-chain MCMC.
marg_roi = idata_h.adm_marginals["roi_m"]
draws_roi = idata_h.posterior["roi_m"].values.reshape(-1, len(channels))
mcmc_roi = mmm_mcmc.inference_data.posterior["roi_m"].values.reshape(
    -1, len(channels)
)
interval_rows = []
for i, ch in enumerate(channels):
  interval_rows.append({
      "Channel": ch,
      "True ROI": float(true_params["roi_m"][i]),
      "ADM Mean": float(marg_roi.sel(metric="mean")[i]),
      "ADM 90% CI (marginal)": (
          f"[{float(marg_roi.sel(metric='ci_lo')[i]):.2f},"
          f" {float(marg_roi.sel(metric='ci_hi')[i]):.2f}]"
      ),
      "ADM 90% CI (1k draws)": (
          f"[{np.quantile(draws_roi[:, i], 0.05):.2f},"
          f" {np.quantile(draws_roi[:, i], 0.95):.2f}]"
      ),
      "MCMC Mean": float(np.mean(mcmc_roi[:, i])),
      "MCMC 90% CI": (
          f"[{np.quantile(mcmc_roi[:, i], 0.05):.2f},"
          f" {np.quantile(mcmc_roi[:, i], 0.95):.2f}]"
      ),
  })
print("\n=== Channel ROI: 90% Credible Intervals (ADM Hessian vs. Full MCMC) ===")
print(pd.DataFrame(interval_rows).round(3).to_string(index=False))

# The approximate draws flow through the Analyzer exactly like MCMC draws, so
# every summary metric now carries a credible interval instead of a point.
analyzer_adm_h = analyzer.Analyzer(
    model_context=mmm_adm_h.model_context, inference_data=idata_h
)
roi_ci_h = (
    analyzer_adm_h.summary_metrics()["roi"]
    .sel(distribution=constants.POSTERIOR)
    .to_dataframe()["roi"]
    .unstack("metric")
)
print("\n=== Analyzer ROI summary from the 1,000 ADM approximate draws ===")
print(roi_ci_h.round(3).to_string())


ADM + Hessian + 1,000 approximate draws: 33.0s
InferenceData groups: ['posterior', 'sample_stats', 'prior', 'trace', 'adm_marginals', 'point_estimate']
Hessian diagnostics: {'hessian_dimension': 91, 'hessian_min_eigenvalue': 0.0683700961326547, 'hessian_max_eigenvalue': 52205.29809217687, 'hessian_is_positive_definite': True, 'hessian_n_floored_eigenvalues': 0}

=== Channel ROI: 90% Credible Intervals (ADM Hessian vs. Full MCMC) ===
Channel  True ROI  ADM Mean ADM 90% CI (marginal) ADM 90% CI (1k draws)  MCMC Mean  MCMC 90% CI
   ch_0      0.85     1.147          [0.19, 2.77]          [0.19, 2.72]      1.312 [0.26, 3.31]
   ch_1      1.55     1.092          [0.16, 2.72]          [0.18, 2.61]      1.185 [0.25, 2.87]
   ch_2      2.35     1.573          [0.17, 4.17]          [0.22, 4.19]      1.634 [0.30, 4.18]
   ch_3      3.20     2.538          [0.22, 6.98]          [0.22, 6.67]      2.771 [0.38, 7.36]

=== Analyzer ROI summary from the 1,000 ADM approximate draws ===
metric         m

## 6. Downstream Meridian Outputs: Model Fit, Response Curves, Budget Optimization & Model Quality Review

Both `mmm.sample_posterior(method="map")` and `mmm.sample_posterior(method="adm")` populate `mmm.inference_data.posterior` with all required parameter and counterfactual tensors, so **every downstream Meridian module** works seamlessly:
1. **`Analyzer.predictive_accuracy()`**: National & Geo-level `R_Squared`, `MAPE`, and `wMAPE`
2. **`Analyzer.response_curves()` / `adstock_decay()` / `hill_curves()`**
3. **`BudgetOptimizer(mmm).optimize()`**: Optimal budget allocation across channels
4. **`mmm.review()` (`ModelReviewer`)**: Automated model health checks (`ConvergenceCheck`, `BayesianPPPCheck`, `GoodnessOfFitCheck`, `PriorPosteriorShiftCheck`, `ROIConsistencyCheck`, `NegativeBaselineCheck`, `HighVarianceCheck`)
5. **`Visualizer.MediaSummary(mmm).summary_table()`**

In [7]:
# 1. Predictive Accuracy (R_Squared, MAPE, wMAPE)
acc_map = (
    analyzer_map.predictive_accuracy()[constants.VALUE]
    .to_dataframe()
    .reset_index()
    .rename(columns={constants.VALUE: "Plain MAP"})
)
acc_adm = (
    analyzer_adm.predictive_accuracy()[constants.VALUE]
    .to_dataframe()
    .reset_index()
    .rename(columns={constants.VALUE: "ADM"})
)
acc_mcmc = (
    analyzer_mcmc.predictive_accuracy()[constants.VALUE]
    .to_dataframe()
    .reset_index()
    .rename(columns={constants.VALUE: "Full MCMC"})
)
acc_df = acc_map.merge(
    acc_adm, on=[constants.METRIC, constants.GEO_GRANULARITY]
).merge(acc_mcmc, on=[constants.METRIC, constants.GEO_GRANULARITY])
print("=== Predictive Accuracy (Model Fit) Comparison ===")
print(acc_df.to_string(index=False))

# 2. Response Curves
rc_map = analyzer_map.response_curves()
rc_adm = analyzer_adm.response_curves()
rc_mcmc = analyzer_mcmc.response_curves()

# 3. Budget Optimization (Fixed Budget Scenario) on Plain MAP vs. ADM vs. Full MCMC
t0 = time.perf_counter()
opt_map = optimizer.BudgetOptimizer(mmm_map).optimize(use_kpi=False)
opt_map_time = time.perf_counter() - t0

t0 = time.perf_counter()
opt_adm = optimizer.BudgetOptimizer(mmm_adm).optimize(use_kpi=False)
opt_adm_time = time.perf_counter() - t0

t0 = time.perf_counter()
opt_mcmc = optimizer.BudgetOptimizer(mmm_mcmc).optimize(use_kpi=False)
opt_mcmc_time = time.perf_counter() - t0

opt_rows = []
for ch in channels:
  hist_spend = float(
      opt_adm.nonoptimized_data.sel(channel=ch)[constants.SPEND].values
  )
  map_opt_spend = float(
      opt_map.optimized_data.sel(channel=ch)[constants.SPEND].values
  )
  adm_opt_spend = float(
      opt_adm.optimized_data.sel(channel=ch)[constants.SPEND].values
  )
  mcmc_opt_spend = float(
      opt_mcmc.optimized_data.sel(channel=ch)[constants.SPEND].values
  )
  map_opt_inc = float(
      opt_map.optimized_data.sel(channel=ch, metric=constants.MEAN)[
          constants.INCREMENTAL_OUTCOME
      ].values
  )
  adm_opt_inc = float(
      opt_adm.optimized_data.sel(channel=ch, metric=constants.MEAN)[
          constants.INCREMENTAL_OUTCOME
      ].values
  )
  mcmc_opt_inc = float(
      opt_mcmc.optimized_data.sel(channel=ch, metric=constants.MEAN)[
          constants.INCREMENTAL_OUTCOME
      ].values
  )
  opt_rows.append({
      "Channel": ch,
      "Historical Spend": hist_spend,
      "Plain MAP Opt Spend": map_opt_spend,
      "ADM Opt Spend": adm_opt_spend,
      "MCMC Opt Spend": mcmc_opt_spend,
      "Plain MAP Opt Inc": map_opt_inc,
      "ADM Opt Inc": adm_opt_inc,
      "MCMC Opt Inc": mcmc_opt_inc,
  })
opt_df = pd.DataFrame(opt_rows)
print(
    f"\n=== Budget Optimization Comparison (MAP: {opt_map_time:.2f}s | ADM:"
    f" {opt_adm_time:.2f}s | MCMC: {opt_mcmc_time:.2f}s) ==="
)
print(opt_df.to_string(index=False))

# 4. Automated Model Quality Review (mmm.review())
review_map = mmm_map.review()
review_adm = mmm_adm.review()
review_mcmc = mmm_mcmc.review()
print("\n=== Model Quality Review: Plain MAP ===")
print(review_map)
print("\n=== Model Quality Review: ADM ===")
print(review_adm)
print("\n=== Model Quality Review: Full MCMC ===")
print(review_mcmc)

# 5. Visualizer MediaSummary Table (ADM Posterior)
ms_adm = visualizer.MediaSummary(mmm_adm)
print("\n=== Visualizer MediaSummary Table (ADM Posterior) ===")
print(
    ms_adm.summary_table(include_prior=False, include_posterior=True).to_string(
        index=False
    )
)

=== Predictive Accuracy (Model Fit) Comparison ===
   metric geo_granularity  Plain MAP      ADM  Full MCMC
R_Squared             geo   0.908282 0.908573   0.906121
R_Squared        national   0.951267 0.950619   0.948870
     MAPE             geo   0.014354 0.014343   0.014464
     MAPE        national   0.004958 0.004979   0.005067
    wMAPE             geo   0.014307 0.014299   0.014412
    wMAPE        national   0.004948 0.004968   0.005054

=== Budget Optimization Comparison (MAP: 2.75s | ADM: 1.61s | MCMC: 8.39s) ===
Channel  Historical Spend  Plain MAP Opt Spend  ADM Opt Spend  MCMC Opt Spend  Plain MAP Opt Inc  ADM Opt Inc  MCMC Opt Inc
   ch_0             997.0                965.0          789.0           838.0         548.450672  1002.912092   1165.128501
   ch_1            1031.0                914.0          744.0           722.0         546.743753   963.224122   1081.597414
   ch_2            1000.0               1169.0         1199.0          1172.0         675.861815  

## 7. Visual Comparison Dashboard (Full MCMC vs. Plain MAP vs. ADM)

The 6-panel dashboard below compares **Full MCMC**, **Plain MAP** (mode), and **ADM** (mean-targeted):
1. **Wall-Clock Runtime (s)**: ADM & Plain MAP (Warm/Cold) vs. Full MCMC (4 chains, 4,000 kept draws).
2. **Channel ROI (`roi_m`)**: Ground Truth vs. Plain MAP vs. ADM vs. Full MCMC Posterior Mean & 90% CI.
3. **Channel Contribution % (`pct_of_contribution`)**: Plain MAP vs. ADM vs. Full MCMC Posterior Mean & 90% CI.
4. **Nonlinear Adstock Decay (`alpha_m`)**: Ground Truth vs. Plain MAP vs. ADM vs. Full MCMC Mean.
5. **Standardized Parameter Error (`|Estimate - MCMC Mean| / Posterior SD`)**: Plain MAP vs. ADM across `roi_m`, `beta_m`, `alpha_m`, `ec_m`, `eta_m`, `xi_c`, and `tau_g`.
6. **Channel Response Curves (`response_curves()`)**: Plain MAP (dotted) vs. ADM (solid) vs. Full MCMC Mean (dashed) across spend multipliers `[0.0, 2.0]`.

In [2]:
fig, axes = plt.subplots(2, 3, figsize=(19, 11))

# Panel 1: Wall-Clock Runtime Comparison
ax = axes[0, 0]
methods_labels = [
    "ADM\n(1-Restart Warm)",
    "ADM\n(5-Restart Warm)",
    "Plain MAP\n(5-Restart Warm)",
    "ADM\n(5-Restart Cold)",
    "Full MCMC NUTS\n(4 Chains, 4000 Draws)",
]
times_vals = [
    adm_warm_1restart_seconds,
    adm_warm_5restarts_seconds,
    map_warm_5restarts_seconds,
    adm_cold_seconds,
    mcmc_seconds,
]
colors = ["#1b9e77", "#2b8cbe", "#7570b3", "#4eb3d3", "#d95f02"]
bars = ax.barh(
    methods_labels, times_vals, color=colors, edgecolor="black", alpha=0.9
)
ax.set_xscale("log")
ax.set_xlabel("Wall-Clock Runtime (seconds, log scale)")
ax.set_title(
    "A. Wall-Clock Runtime (ADM vs. Plain MAP vs. MCMC)", fontweight="bold"
)
for bar, val in zip(bars, times_vals):
  ax.text(
      val * 1.12,
      bar.get_y() + bar.get_height() / 2,
      f"{val:.2f}s ({mcmc_seconds / val:.1f}x)",
      va="center",
      fontsize=8.5,
      fontweight="bold",
  )
ax.set_xlim(right=max(times_vals) * 4.5)

# Panel 2: Channel ROI Comparison (Ground Truth vs Plain MAP vs ADM vs Full MCMC)
ax = axes[0, 1]
x = np.arange(len(channels))
width = 0.20
true_rois = [true_params["roi_m"][i] for i in range(len(channels))]
map_rois = [
    float(
        summary_map[constants.ROI]
        .sel(
            channel=ch, metric=constants.MEAN, distribution=constants.POSTERIOR
        )
        .values
    )
    for ch in channels
]
adm_rois = [
    float(
        summary_adm[constants.ROI]
        .sel(
            channel=ch, metric=constants.MEAN, distribution=constants.POSTERIOR
        )
        .values
    )
    for ch in channels
]
mcmc_rois = [
    float(
        summary_mcmc[constants.ROI]
        .sel(
            channel=ch, metric=constants.MEAN, distribution=constants.POSTERIOR
        )
        .values
    )
    for ch in channels
]
mcmc_ci_lo = [
    float(
        summary_mcmc[constants.ROI]
        .sel(
            channel=ch, metric=constants.CI_LO, distribution=constants.POSTERIOR
        )
        .values
    )
    for ch in channels
]
mcmc_ci_hi = [
    float(
        summary_mcmc[constants.ROI]
        .sel(
            channel=ch, metric=constants.CI_HI, distribution=constants.POSTERIOR
        )
        .values
    )
    for ch in channels
]
yerr = [
    np.maximum(0, np.array(mcmc_rois) - np.array(mcmc_ci_lo)),
    np.maximum(0, np.array(mcmc_ci_hi) - np.array(mcmc_rois)),
]
ax.bar(
    x - 1.5 * width,
    true_rois,
    width,
    label="Ground Truth",
    color="#636363",
    alpha=0.85,
)
ax.bar(
    x - 0.5 * width,
    map_rois,
    width,
    label="Plain MAP (Mode)",
    color="#7570b3",
    alpha=0.9,
)
ax.bar(
    x + 0.5 * width,
    adm_rois,
    width,
    label="ADM (Mean-Targeted)",
    color="#1b9e77",
    alpha=0.9,
)
ax.bar(
    x + 1.5 * width,
    mcmc_rois,
    width,
    yerr=yerr,
    capsize=3.5,
    label="Full MCMC Mean (90% CI)",
    color="#d95f02",
    alpha=0.9,
)
ax.set_xticks(x)
ax.set_xticklabels(channels)
ax.set_ylabel("ROI")
ax.set_title(
    "B. Channel ROI: Plain MAP vs. ADM vs. Full MCMC", fontweight="bold"
)
ax.legend(fontsize=8)

# Panel 3: Channel Contribution % Comparison
ax = axes[0, 2]
w3 = 0.25
map_contribs = [
    float(
        summary_map[constants.PCT_OF_CONTRIBUTION]
        .sel(
            channel=ch, metric=constants.MEAN, distribution=constants.POSTERIOR
        )
        .values
    )
    for ch in channels
]
adm_contribs = [
    float(
        summary_adm[constants.PCT_OF_CONTRIBUTION]
        .sel(
            channel=ch, metric=constants.MEAN, distribution=constants.POSTERIOR
        )
        .values
    )
    for ch in channels
]
mcmc_contribs = [
    float(
        summary_mcmc[constants.PCT_OF_CONTRIBUTION]
        .sel(
            channel=ch, metric=constants.MEAN, distribution=constants.POSTERIOR
        )
        .values
    )
    for ch in channels
]
mcmc_c_lo = [
    float(
        summary_mcmc[constants.PCT_OF_CONTRIBUTION]
        .sel(
            channel=ch, metric=constants.CI_LO, distribution=constants.POSTERIOR
        )
        .values
    )
    for ch in channels
]
mcmc_c_hi = [
    float(
        summary_mcmc[constants.PCT_OF_CONTRIBUTION]
        .sel(
            channel=ch, metric=constants.CI_HI, distribution=constants.POSTERIOR
        )
        .values
    )
    for ch in channels
]
yerr_c = [
    np.maximum(0, np.array(mcmc_contribs) - np.array(mcmc_c_lo)),
    np.maximum(0, np.array(mcmc_c_hi) - np.array(mcmc_contribs)),
]
ax.bar(
    x - w3,
    map_contribs,
    w3,
    label="Plain MAP (Mode)",
    color="#7570b3",
    alpha=0.9,
)
ax.bar(
    x, adm_contribs, w3, label="ADM (Mean-Targeted)", color="#1b9e77", alpha=0.9
)
ax.bar(
    x + w3,
    mcmc_contribs,
    w3,
    yerr=yerr_c,
    capsize=3.5,
    label="Full MCMC Mean (90% CI)",
    color="#d95f02",
    alpha=0.9,
)
ax.set_xticks(x)
ax.set_xticklabels(channels)
ax.set_ylabel("% of Total Outcome")
ax.set_title(
    "C. Channel Contribution % (Plain MAP vs. ADM vs. MCMC)", fontweight="bold"
)
ax.legend(fontsize=8)

# Panel 4: Nonlinear Adstock Decay (alpha_m) Recovery
ax = axes[1, 0]
ax.plot(
    x,
    nonlinear_df["alpha_m (True)"],
    "s--",
    color="#636363",
    label="alpha_m Ground Truth",
)
ax.plot(
    x,
    nonlinear_df["alpha_m (Plain MAP)"],
    "v:",
    color="#7570b3",
    linewidth=2,
    label="alpha_m Plain MAP (Mode)",
)
ax.plot(
    x,
    nonlinear_df["alpha_m (ADM)"],
    "o-",
    color="#1b9e77",
    linewidth=2.2,
    label="alpha_m ADM (Beta Adj)",
)
ax.plot(
    x,
    nonlinear_df["alpha_m (MCMC Mean)"],
    "^-",
    color="#d95f02",
    linewidth=2,
    label="alpha_m Full MCMC Mean",
)
ax.set_xticks(x)
ax.set_xticklabels(channels)
ax.set_ylabel("Adstock Decay (alpha_m)")
ax.set_title(
    "D. Adstock Decay alpha_m (Plain MAP vs. ADM vs. MCMC)", fontweight="bold"
)
ax.legend(fontsize=8)

# Panel 5: Standardized Parameter Error |Estimate - MCMC Mean| / Posterior SD
ax = axes[1, 1]
key_params = ["roi_m", "beta_m", "alpha_m", "ec_m", "eta_m", "xi_c", "tau_g"]
sub_df = param_summary_df.set_index("Parameter").loc[key_params]
xp = np.arange(len(key_params))
wp = 0.36
ax.bar(
    xp - wp / 2,
    sub_df["Plain MAP |Err|/SD"],
    wp,
    label="Plain MAP (Mode)",
    color="#7570b3",
    alpha=0.9,
)
ax.bar(
    xp + wp / 2,
    sub_df["ADM |Err|/SD"],
    wp,
    label="ADM (Mean-Targeted)",
    color="#1b9e77",
    alpha=0.9,
)
ax.set_xticks(xp)
ax.set_xticklabels(key_params, rotation=20)
ax.set_ylabel("Mean |Est - MCMC Mean| / Posterior SD")
ax.set_title(
    "E. Standardized Error vs. Full MCMC Mean (Lower is Better)",
    fontweight="bold",
)
ax.legend(fontsize=8.5)

# Panel 6: Response Curves (Plain MAP vs. ADM vs. Full MCMC Mean)
ax = axes[1, 2]
palette = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728"]
for idx, ch in enumerate(channels):
  mult = rc_adm.spend_multiplier.values
  inc_map_curve = (
      rc_map[constants.INCREMENTAL_OUTCOME]
      .sel(channel=ch, metric=constants.MEAN)
      .values
  )
  inc_adm_curve = (
      rc_adm[constants.INCREMENTAL_OUTCOME]
      .sel(channel=ch, metric=constants.MEAN)
      .values
  )
  inc_mcmc_curve = (
      rc_mcmc[constants.INCREMENTAL_OUTCOME]
      .sel(channel=ch, metric=constants.MEAN)
      .values
  )
  ax.plot(
      mult,
      inc_adm_curve,
      color=palette[idx],
      linestyle="-",
      linewidth=2.2,
      label=f"{ch} (ADM)",
  )
  ax.plot(
      mult,
      inc_mcmc_curve,
      color=palette[idx],
      linestyle="--",
      linewidth=1.8,
      label=f"{ch} (MCMC)",
  )
  ax.plot(
      mult,
      inc_map_curve,
      color=palette[idx],
      linestyle=":",
      linewidth=1.8,
      alpha=0.90,
      label=f"{ch} (Plain MAP)",
  )
ax.set_xlabel("Spend Multiplier")
ax.set_ylabel("Incremental Outcome")
ax.set_title(
    "F. Response Curves (Solid=ADM, Dashed=MCMC, Dotted=Plain MAP)",
    fontweight="bold",
)
ax.legend(fontsize=6.8, ncol=3)

plt.tight_layout()
fig.savefig(
    "meridian_full_map_vs_mcmc.png",
    dpi=150,
    bbox_inches="tight",
)
plt.show()